<a href="https://colab.research.google.com/github/lbenit/Floristic_map_africa/blob/main/04_Extract_predictors_to_plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/lbenit/Floristic_map_africa.git


Cloning into 'Floristic_map_africa'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 1.35 MiB | 2.70 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [ ]:
%cd Floristic_map_africa


/content/Floristic_map_africa/Floristic_map_africa


In [ ]:
# Print working directory
import os
print(os.getcwd())

/content/Floristic_map_africa/Floristic_map_africa


# Connect to earth engine

In [2]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-') # connect to account

In [3]:
import pandas as pd
import numpy as np
import geemap
import matplotlib.pyplot as plt

# Load data

In [5]:
plots=pd.read_csv('selected_seosaw_plots.csv')

In [6]:
plots.head()

,plot_id,country_iso3,plot_shape,plot_area,latitude_of_centre,longitude_of_centre,census_date
0,MAR_1,MOZ,circle,0.125664,-15.271106,36.485275,2015
1,MAR_2,MOZ,circle,0.125664,-15.269273,36.485329,2015
2,MAR_3,MOZ,circle,0.125664,-15.282653,36.488617,2015
3,MAR_4,MOZ,circle,0.125664,-15.280858,36.488645,2015
4,MAR_5,MOZ,circle,0.125664,-15.272123,36.501926,2015


# Make into feature collection

In [7]:
# Convert to feature collection
def df_to_fc(df):
    features = []
    for _, row in df.iterrows():
        props = {k: (v.item() if hasattr(v, "item") else v) for k, v in row.to_dict().items()}
        point = ee.Geometry.Point([props['longitude_of_centre'], props['latitude_of_centre']])
        feat = ee.Feature(point, props)
        features.append(feat)
    return ee.FeatureCollection(features)



In [8]:
data= df_to_fc(plots)

In [9]:
# Save plots as asset
# Save as asset
task = ee.batch.Export.table.toAsset(
    collection=data,
    description='filtered_plots',
    assetId='projects/ee-benitezl/assets/Floristics/review/filtered_plots_all'
)
task.start()

# Get imagery
This involves extracting values from sentinel-2 and Modis

In [10]:
# Load in area of interest
landscape = ee.Image('projects/ee-benitezl/assets/10km_roi')

In [11]:
# Get EVI
def addEVI(image):
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }
    ).rename('EVI')
    return image.addBands(evi)

In [16]:
# Get Sentinel 2 first set of bands
# I split it up because it becomes quite large with all of the bands together
s2_2020_1 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate('2020-01-01', '2021-01-01')
    .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 70)
    .map(lambda im: im.updateMask(im.select('SCL').lte(6).And(im.select('SCL').gte(4))))
    .map(addEVI)
    .select(['EVI','B9','B11','B12','B1','B2',])
)

In [17]:
# Get Sentinel 2 second set of bands
s2_2020_2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate('2020-01-01', '2021-01-01')
    .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 70)
    .map(lambda im: im.updateMask(im.select('SCL').lte(6).And(im.select('SCL').gte(4))))
    .map(addEVI)
    .select(['B3','B4','B5','B6','B7','B8'])
)

In [18]:
# Reduce Sentinel 2
reducer = (
    ee.Reducer.mean()
    .combine(ee.Reducer.median(), '', True)
    .combine(ee.Reducer.max(), '', True)
    .combine(ee.Reducer.min(), '', True)
    .combine(ee.Reducer.stdDev(), '', True)
    .combine(ee.Reducer.percentile([5]), '', True)
    .combine(ee.Reducer.percentile([95]), '', True)
)

results_2020_1 = s2_2020_1.reduce(reducer)
results_2020_2 = s2_2020_2.reduce(reducer)

In [33]:
# Get phenology from MODIS
vipphen = ee.ImageCollection('MODIS/061/MCD12Q2').filter(ee.Filter.date('2012-01-01', '2023-01-01')).mean().unmask(0)

season = vipphen.select('EVI_Amplitude_2').gt(vipphen.select('EVI_Amplitude_1'))

phenology_properties = [
    'Greenup','MidGreenup','Peak','Maturity','Senescence',
    'MidGreendown','Dormancy','EVI_Minimum','EVI_Amplitude','EVI_Area'
]

def getPhenology(season):
    bands = [p + '_' + season for p in phenology_properties]
    return vipphen.select(bands).rename(phenology_properties)

phenology = getPhenology('1').where(season, getPhenology('2'))

In [36]:
# Get climate and topography data
worldClim = ee.Image("WORLDCLIM/V1/BIO")
tpi = ee.Image("CSP/ERGo/1_0/Global/SRTM_mTPI").rename('tpi')
hnd = ee.Image("MERIT/Hydro/v1_0_1").select('hnd')#97m
elevation = ee.Image("USGS/SRTMGL1_003").select('elevation')

# Combine layers

In [38]:
# Rename layers
im1a = results_2020_1
im1b = results_2020_2

In [39]:
# Combine layers
im2 = elevation.addBands(worldClim).addBands(phenology).addBands(tpi).addBands(hnd)

# Load in saved imagery and extract data to points

In [23]:
data=ee.FeatureCollection('projects/ee-benitezl/assets/Floristics/review/filtered_plots_all')

In [26]:
# Make landscape mask
masked = landscape.updateMask(landscape.eq(1))



In [27]:
# Filter points to remove any outside of landscape=1
filtered_points = masked.sampleRegions(
    collection=data,
    geometries=True
)


In [28]:
# Check how many points are in filtered
filtered_points.size().getInfo()

251

In [ ]:
# Save filter points to drive
task = ee.batch.Export.table.toDrive(
    collection=filtered_points,
    description='filtered_plots',
    fileFormat='CSV'
)
task.start()

# Extract data to plots

In [29]:
data=filtered_points

In [41]:
samples1a = im1a.sampleRegions(
    collection=data,
    scale=10,
    geometries=False
)


In [42]:
samples1b = im1b.sampleRegions(
    collection=data,
    scale=10,
    geometries=False
)

In [43]:
samples2 = im2.sampleRegions(
    collection=data,
    scale=30,
    geometries=False
)


In [44]:
task1a = ee.batch.Export.table.toDrive(
    collection=samples1a,
    description='extracted_predictors_all_plots_1a',
    fileFormat='CSV'
)
task1a.start()


In [45]:
task1b = ee.batch.Export.table.toDrive(
    collection=samples1b,
    description='extracted_predictors_all_plots_1b',
    fileFormat='CSV'
)
task1b.start()

In [46]:
task2 = ee.batch.Export.table.toDrive(
    collection=samples2,
    description='extracted_predictors_all_plots_2',
    fileFormat='CSV'
)
task2.start()

In [ ]:
task2.status()

# Soil data

In [47]:

# ------------------------------------------------------------
# 2. Map of iSDAsoil variables → Earth Engine asset IDs
#    (extend or tweak as needed)
# ------------------------------------------------------------
assets = {
    "ph":                 "ISDASOIL/Africa/v1/ph",
    "soc":                "ISDASOIL/Africa/v1/carbon_organic",
    "sand":               "ISDASOIL/Africa/v1/sand_content",
    "silt":               "ISDASOIL/Africa/v1/silt_content",
    "clay":               "ISDASOIL/Africa/v1/clay_content",
    "bd":                 "ISDASOIL/Africa/v1/bulk_density",
    "cec":                "ISDASOIL/Africa/v1/cation_exchange_capacity",
    "nitrogen_total":     "ISDASOIL/Africa/v1/nitrogen_total",
    "phosphorus_extractable": "ISDASOIL/Africa/v1/phosphorus_extractable",
    "potassium_extractable":  "ISDASOIL/Africa/v1/potassium_extractable",
    "calcium_extractable":    "ISDASOIL/Africa/v1/calcium_extractable",
    "magnesium_extractable":  "ISDASOIL/Africa/v1/magnesium_extractable",
    "sulfur_extractable":     "ISDASOIL/Africa/v1/sulphur_extractable",
    "iron_extractable":       "ISDASOIL/Africa/v1/iron_extractable",
    "zinc_extractable":       "ISDASOIL/Africa/v1/zinc_extractable"
}

depth_bands = [
    'mean_0_20',
    'mean_20_50',
    'stdev_0_20',
    'stdev_20_50'
]

depth_suffixes = [
    'mean_0_20',
    'mean_20_50',
    'stdev_0_20',
    'stdev_20_50'
]


images = []

for short_name, asset_id in assets.items():
    try:
        img = ee.Image(asset_id)
        # Select the three depth bands and rename them to shortname_depth
        renamed = img.select(depth_bands).rename(
            [f"{short_name}_{suf}" for suf in depth_suffixes]
        )
        images.append(renamed)
    except Exception as e:
        print(f"Skipping {asset_id}: {e}")

# ------------------------------------------------------------
# 3. Stack all variables into one multiband image
# ------------------------------------------------------------
isda_stack = ee.Image.cat(images)
print("Bands:", isda_stack.bandNames().getInfo())

# ------------------------------------------------------------
# 4. Sample at point locations
# ------------------------------------------------------------
sampled = isda_stack.sampleRegions(
    collection=data,
    properties=['plot_id'],
    scale=30,
    geometries=False
)

print("First sampled feature:", sampled.first().getInfo())



Bands: ['ph_mean_0_20', 'ph_mean_20_50', 'ph_stdev_0_20', 'ph_stdev_20_50', 'soc_mean_0_20', 'soc_mean_20_50', 'soc_stdev_0_20', 'soc_stdev_20_50', 'sand_mean_0_20', 'sand_mean_20_50', 'sand_stdev_0_20', 'sand_stdev_20_50', 'silt_mean_0_20', 'silt_mean_20_50', 'silt_stdev_0_20', 'silt_stdev_20_50', 'clay_mean_0_20', 'clay_mean_20_50', 'clay_stdev_0_20', 'clay_stdev_20_50', 'bd_mean_0_20', 'bd_mean_20_50', 'bd_stdev_0_20', 'bd_stdev_20_50', 'cec_mean_0_20', 'cec_mean_20_50', 'cec_stdev_0_20', 'cec_stdev_20_50', 'nitrogen_total_mean_0_20', 'nitrogen_total_mean_20_50', 'nitrogen_total_stdev_0_20', 'nitrogen_total_stdev_20_50', 'phosphorus_extractable_mean_0_20', 'phosphorus_extractable_mean_20_50', 'phosphorus_extractable_stdev_0_20', 'phosphorus_extractable_stdev_20_50', 'potassium_extractable_mean_0_20', 'potassium_extractable_mean_20_50', 'potassium_extractable_stdev_0_20', 'potassium_extractable_stdev_20_50', 'calcium_extractable_mean_0_20', 'calcium_extractable_mean_20_50', 'calcium_

In [48]:


# Export to Drive (optional)
task = ee.batch.Export.table.toDrive(
    collection=sampled,
    description="isda_soil_data",
    fileFormat="CSV"
)
task.start()


# Fire data

The fire data was derived using the following code in earth engine (not through the python API)





// 1. Load MODIS MCD64A1.061 Burned Area Collection

var collection = ee.ImageCollection('MODIS/061/MCD64A1')
  .filterDate('2005-01-01', '2025-01-01');


// 2. Extract the burn date band
//    burn_date > 0 means the pixel burned in that month
//    burn_date is the day-of-year of burning


var burnDate = collection.select('BurnDate');


// 3. Convert burn_date to a binary burned/not-burned mask


var burnedMask = burnDate.map(function(img) {
  var burned = img.gt(0);  // burned pixels = 1
  return burned.rename('burned').copyProperties(img, img.propertyNames());
});


// 4. Sum all burned months to get fire frequency
//    (number of times the pixel burned)


var fireFrequency = burnedMask.sum().rename('fire_frequency');


// 5. Replace NA (masked) pixels with zero


var fireFrequencyZero = fireFrequency.unmask(0);


// 6. Display


Map.centerObject(fireFrequencyZero, 5);
Map.addLayer(
  fireFrequencyZero,
  {min: 0, max: 10, palette: ['white', 'yellow', 'orange', 'red', 'darkred']},
  'Fire Frequency (NA = 0)'
);


// 6. Export if needed


 Export.image.toAsset({
   image: fireFrequencyZero,
   description: 'FireFrequency_MODIS_2005_2025',
   scale: 500,
   region: geometry,
   maxPixels: 1e13
 });


In [34]:
# Fire data from MODIS
fire= ee.Image('projects/ee-benitezl/assets/FireFrequency_MODIS_2005_2025')

In [49]:
# Extract fire data to plots
fire_data = fire.sampleRegions(
    collection=data,
    properties=['plot_id'],
    scale=500,
    geometries=False
)

In [50]:
# Export to Drive (optional)
task = ee.batch.Export.table.toDrive(
    collection=fire_data,
    description="fire_days",
    fileFormat="CSV"
)
task.start()


In [ ]:
task.status()

{'state': 'RUNNING',
 'description': 'extracted_predictors_all_plots_2_new',
 'priority': 100,
 'creation_timestamp_ms': 1785494890393,
 'update_timestamp_ms': 1785494969944,
 'start_timestamp_ms': 1785494899642,
 'task_type': 'EXPORT_FEATURES',
 'attempt': 1,
 'batch_eecu_usage_seconds': 6239.963378906,
 'id': 'CPWRJ5QT77IYWRZSHDB7VFNI',
 'name': 'projects/ee-benitezl/operations/CPWRJ5QT77IYWRZSHDB7VFNI'}